### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] !pip install langchain-community
# [PATCHED] !pip install sentence-transformers
# [PATCHED] !pip install faiss-cpu
# [PATCHED] !pip install llama-cpp-python

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
working_folder='./'

In [ ]:
model_path = working_folder + "model/Meta-Llama-3-8B-Instruct-Q4_K_M.gguf"

In [ ]:
data_folder = working_folder + "data/"

In [ ]:
db_file_name= data_folder + "Hotel_faiss_DB"

In [ ]:
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path=model_path,
    temperature=0.01,
    max_tokens=50,
    top_p=0.95
)

In [ ]:
template = """Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Context: {context}
Question: {question}
Only return the helpful answer below and nothing else.
Helpful answer:
"""

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="distiluse-base-multilingual-cased-v1",
    model_kwargs={'device': str(device)})

from langchain.vectorstores import FAISS

db_store = FAISS.load_local(db_file_name, embeddings, allow_dangerous_deserialization=True)

retriever = db_store.as_retriever(search_kwargs={'k': 2})

from langchain import PromptTemplate
prompt = PromptTemplate(
    template=template,
    input_variables=['context', 'question'])

from langchain.chains import RetrievalQA
qa_llm = RetrievalQA.from_chain_type(llm=llm,
                                     chain_type='stuff',
                                     retriever=retriever,
                                     return_source_documents=True,
                                     chain_type_kwargs={'prompt': prompt})

In [ ]:
def find_line_with_alpha_num(s):

    lines = s.splitlines()

    for line in lines:
        if any(char.isalnum() for char in line):
            return line

    return ""

In [ ]:
question = "Where is the 4Z Hotel is located?"

output = qa_llm.invoke(question)
first_line=find_line_with_alpha_num(output["result"])

print(first_line)

In [ ]:
question = "What is the phone number of the 4Z Hotel?"

output = qa_llm.invoke(question)
first_line=find_line_with_alpha_num(output["result"])

print(first_line)

In [ ]:
question = "هل يوجد مسبح في فندق الفورزد؟ أجب بالعربية"

output = qa_llm.invoke(question)
first_line=find_line_with_alpha_num(output["result"])

print(first_line)

In [ ]:
question = "ماأسعار الغرف في فندق الفورزد؟ "

output = qa_llm.invoke(question)
first_line=find_line_with_alpha_num(output["result"])

print(first_line)